# 01 · Load the data

**Project:** _Science of science_ · **Team:** _Albert Fontbuté, Alba Sentoll, Carlos Jové, Adrià Rojo_

Downloads the raw data once, checks it, and caches it as Parquet in `data/cache/` (git-ignored). Re-run it only when you want fresh data, and say so in the README.

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv(dotenv_path=find_dotenv())                                   # tokens live in ../.env, never in the notebook

True

## Metadata

Name, publisher, licence and last update of each dataset (see `03_open_data_apis` in the course repository).

In [3]:
DATASET = "mrus-ixpq"   # Socrata id of your dataset https://analisi.transparenciacatalunya.cat/api/v3/views/mrus-ixpq/query.json
meta = requests.get(f"https://analisi.transparenciacatalunya.cat/api/views/{DATASET}.json", timeout=30).json()
meta["name"], meta.get("attribution"), meta.get("license")

('Publicacions de recerca de Catalunya',
 'Departament de Recerca i Universitats',
 {'name': 'See Terms of Use'})

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from sodapy import Socrata
from tqdm import tqdm

MAX_WORKERS = 4
MAX_RETRIES = 3
RETRY_BACKOFF = 5  # segundos base (5, 10, 20...)

def fetch_papers_from_year(client, year):
    """Descarga y cachea los papers de un año (síncrono)."""
    data = client.get_all(
        DATASET,
        select='date, doi, ruct, orcidResearchers, orcidDirectors, orcidUncheckedAuthors, orcidUncheckedDirectors',
        where=f'''doi IS NOT NULL 
            AND issn IS NOT NULL 
            AND orcidResearchers IS NOT NULL 
            AND ruct IS NOT NULL 
            AND date BETWEEN "{year}-01-01T00:00:00" AND "{year+1}-01-01T00:00:00" ''',
    )
    df = pd.DataFrame(data)
    if len(df) > 0:
        df = df.sort_values(by='date', ignore_index=True)
        df.to_parquet(path=f'../data/cache/papers_{year}.parquet')
    return year, len(df)


def fetch_with_retry(client, year):
    """Reintenta con backoff exponencial ante 429 / errores transitorios."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fetch_papers_from_year(client, year)
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF * (2 ** (attempt - 1)))


def main():
    CACHE.mkdir(parents=True, exist_ok=True)
    years = list(range(1949, 2027))

    pending = [y for y in years if not (CACHE / f"papers_{y}.parquet").exists()]
    skipped = len(years) - len(pending)
    if skipped:
        print(f"{skipped} años ya en caché, {len(pending)} por descargar.")

    if not pending:
        return

    with Socrata(
        domain="analisi.transparenciacatalunya.cat",
        app_token=os.environ['SOCRATA_APP_TOKEN'],
        username=os.environ['SOCRATA_USER_ID'],
        password=os.environ['SOCRATA_USER_PASSWD'],
        timeout=600,
    ) as client, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:

        futures = {ex.submit(fetch_with_retry, client, y): y for y in pending}

        errors = []
        with tqdm(total=len(futures), desc="Descargando papers", unit="año") as pbar:
            for fut in as_completed(futures):
                year = futures[fut]
                try:
                    _, n = fut.result()
                    pbar.set_postfix_str(f"{year}: {n} filas")
                except Exception as e:
                    errors.append((year, e))
                    pbar.set_postfix_str(f"{year}: ERROR")
                pbar.update(1)

    if errors:
        print("\nAños con error:")
        for year, err in errors:
            print(f"  {year}: {type(err).__name__}: {err}")


if __name__ == "__main__":
    main()

66 años ya en caché, 12 por descargar.


Descargando papers: 100%|██████████| 12/12 [1:22:56<00:00, 414.73s/año, 2021: 594481 filas]


# Data loading

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv
from tqdm import tqdm

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv(dotenv_path=find_dotenv())       


True

In [5]:
all_papers = pd.DataFrame()
for f in tqdm(list(CACHE.glob("papers_*.parquet"))):
    if (all_papers.empty):
        all_papers = pd.read_parquet(f)
    else:
        all_papers = pd.concat([all_papers, pd.read_parquet(f)], ignore_index=True)

  0%|          | 0/77 [00:00<?, ?it/s]

100%|██████████| 77/77 [00:01<00:00, 57.14it/s] 


In [5]:
all_papers.columns

Index(['date', 'doi', 'ruct', 'orcidResearchers', 'orcidUncheckedAuthors',
       'orcidDirectors'],
      dtype='str')

In [6]:
cleaned_papers = all_papers.drop_duplicates(ignore_index=True)
cleaned_papers['orcidResearchers'] = cleaned_papers['orcidResearchers'].str.split(';')
cleaned_papers['orcidDirectors'] = cleaned_papers['orcidDirectors'].str.split(';')
cleaned_papers['orcidUncheckedAuthors'] = cleaned_papers['orcidUncheckedAuthors'].str.split(';')

cleaned_papers.to_parquet(DATA / 'papers_cleaned.gencat.parquet', index=False)
del all_papers 
len(cleaned_papers)

1747707

In [2]:
cleaned_papers = pd.read_parquet(DATA/'papers_cleaned.gencat.parquet')
cleaned_papers

,date,doi,ruct,orcidResearchers,orcidUncheckedAuthors,orcidDirectors
0,1949-01-01T00:00:00.000,10.1210/jcem-9-9-883,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None
1,1952-01-01T00:00:00.000,10.1002/cber.19520850704,Universitat Rovira i Virgili,[0000-0001-6112-5913],None,None
2,1952-01-01T00:00:00.000,10.1111/j.1365-2133.1952.tb15900.x,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None
3,1953-01-01T00:00:00.000,10.1038/jid.1953.55,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None
4,1954-01-01T00:00:00.000,10.1016/s0022-5347(17)67795-4,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None
...,...,...,...,...,...,...
1747702,2022-01-01T00:00:00.000,10.1017/s104775942200037x,Universitat Autònoma de Barcelona,"[0000-0003-4771-1216, 0000-0002-5942-6992, 000...",None,None
1747703,2022-01-01T00:00:00.000,10.1089/aid.2021.0141,Universitat de Vic - Universitat Central de Ca...,"[0000-0003-3232-4598, 0000-0002-5343-2881, 000...",None,None
1747704,2022-01-01T00:00:00.000,10.3389/fmed.2022.815739,Fundació de Recerca Clínic Barcelona-Institut ...,"[0000-0003-0853-6865, 0000-0001-7330-9621, 000...",None,None
1747705,2022-01-01T00:00:00.000,10.3389/fendo.2022.984877,Institut de Recerca Hospital de Sant Pau – IIB...,"[0000-0003-0475-3096, 0000-0002-8837-4343, 000...",None,None


In [3]:
(
    cleaned_papers['ruct']
    .explode()
    .drop_duplicates()
    .dropna()
    .reset_index(drop=True)
).to_frame().to_csv(CACHE / 'ructs.csv', index=False, header=False)

In [4]:
checked_orcids = (
    pd.concat([cleaned_papers['orcidResearchers'], cleaned_papers['orcidDirectors']])
        .explode()
        .dropna()
        .drop_duplicates()
        .reset_index(drop=True)
)

checked_orcids.name = 'orcid'

checked_orcids.to_frame().to_parquet(CACHE / 'checked_orcids.parquet', index=False)

In [3]:
unchecked_orcids = (
    cleaned_papers['orcidUncheckedAuthors']
        .explode()
            .drop_duplicates()
            .dropna()
            .reset_index(drop=True)
)
unchecked_orcids.name = 'orcid'
unchecked_orcids.to_frame().to_parquet(CACHE / 'unchecked_orcids.parquet', index=False)

In [7]:
cleaned_papers['doi'].drop_duplicates().dropna().to_frame().to_parquet(CACHE/'dois.parquet', index=False)

## Download (all rows) and cache

In [ ]:
def cached(name, fetch):
    """Return data/cache/<name>.parquet; download it with fetch() the first time."""
    path = CACHE / f"{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    df = fetch()
    df.to_parquet(path, index=False)
    return df

# df = cached("raw", lambda: fetch_socrata(DATASET))   # fetch_socrata: see 03_open_data_apis
# print(len(df), "rows")

In [47]:
import pandas as pd
pd.read_parquet(CACHE/'dois.parquet')

,doi
0,10.1016/0022-03968790133-1
1,10.1016/0022-510x(87)90095-5
2,10.1016/s0163-10478790892-2
3,10.1016/0197-01868790035-0
4,10.1016/s0277-53870080875-x
...,...
358034,10.1016/0022-328x(93)83411-n
358035,10.1016/0003-2670(93)85329-i
358036,10.3989/pirineos.1993.v141-142.164
358037,10.3989/collectbot.1993.v22.83


## Checks

Row count against the server, types, missing values.

In [ ]:
def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
    })


# profile(df)

## Dataset card

| Field | Answer |
|---|---|
| Source (URL) and licence | |
| Rows × columns | |
| What is one row? | |
| Columns we use | |
| Missing values / problems | |
| API access? | |